# STORM-PhysNet — Master reproduction notebook

Official companion notebook for the conference + IEEE Access papers.

**What this notebook is for**
1. Load **official** PE tables already in `results/` (15-seed GOES retrain + GRASP transfer).
2. Optional **demo** train of one model/seed with the real `Trainer.fit` API (not full 15×10 campaign).
3. Optional **eval** of one released checkpoint on the GOES test split.
4. Point reviewers to the correct artifacts — not a one-click full retrain.

**Important**
- Headline numbers come from the August 2026 multi-account Kaggle retrain + eval/GRASP scripts.
- Full 15-seed × 10-system training is GPU-heavy; use released `checkpoints/` + `results/*.csv`.
- Default Transformer is **not** architecture-matched (`d_model=64`, 3 layers). Matched control uses `d_model=128`, 2 layers, 4 heads.
- Horizons: **1 h / 6 h / 12 h** (hourly GOES).
- Synthetic data generators are **not** part of the paper pipeline.


## 0. Setup


In [ ]:
import os, sys, subprocess
from pathlib import Path

# ---- environment detection ----
IN_COLAB = Path("/content").exists()
IN_KAGGLE = Path("/kaggle").exists()

if IN_COLAB:
    REPO_DIR = Path("/content/STORM-PhysNet")
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/bnsama29-cloud/STORM-PhysNet.git", str(REPO_DIR)],
            check=True,
        )
    os.chdir(REPO_DIR)
elif IN_KAGGLE:
    REPO_DIR = Path("/kaggle/working/STORM-PhysNet")
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/bnsama29-cloud/STORM-PhysNet.git", str(REPO_DIR)],
            check=True,
        )
    os.chdir(REPO_DIR)
else:
    # local: run from repo root or set REPO_DIR
    REPO_DIR = Path.cwd()
    if not (REPO_DIR / "src").exists():
        # try parent / clone
        cand = REPO_DIR / "STORM-PhysNet"
        if cand.exists():
            REPO_DIR = cand
            os.chdir(REPO_DIR)
        else:
            raise SystemExit("Run from the STORM-PhysNet repo root (must contain src/).")

sys.path.insert(0, str(REPO_DIR.resolve()))
print("REPO_DIR =", REPO_DIR.resolve())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cdflib", "pyyaml", "matplotlib", "pandas"], check=False)

import numpy as np
import pandas as pd
import yaml
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print("torch", torch.__version__, "cuda", torch.cuda.is_available())


## 1. Official paper tables (no training)

These CSVs are the **source of truth** for the manuscripts. They must match the TeX tables.


In [ ]:
RES = REPO_DIR / "results"
assert RES.exists(), f"Missing results/ under {REPO_DIR}"

main = pd.read_csv(RES / "table_main_means.csv")
bagged = pd.read_csv(RES / "table_bagged.csv")
grasp = pd.read_csv(RES / "table_grasp_storm_bz.csv")
params = pd.read_csv(RES / "table_parameter_counts.csv") if (RES / "table_parameter_counts.csv").exists() else None

print("=== Seed-mean PE (15 seeds) ===")
display_cols = [c for c in ["name", "PE_1h", "PE_6h", "PE_12h", "PE_pers_1h", "PE_st_6h"] if c in main.columns]
print(main[display_cols].round(3).to_string(index=False))

print("\n=== True bagged PE (n_members=15) ===")
print(bagged.round(3).to_string(index=False))

print("\n=== GRASP transfer (STORM-Bz, 15 seeds) ===")
print(grasp.to_string(index=False))

if params is not None:
    print("\n=== Parameter counts ===")
    print(params.to_string(index=False))

# Quick consistency checks vs paper headline digits
storm = main.loc[main["name"] == "storm_bz"].iloc[0]
tf = main.loc[main["name"] == "transformer"].iloc[0]
assert abs(storm["PE_1h"] - 0.986) < 0.002, storm["PE_1h"]
assert abs(tf["PE_1h"] - 0.978) < 0.002, tf["PE_1h"]
print("\nConsistency checks against paper headline PE: OK")


## 2. Optional: load GOES data + chronological split

Same pipeline as training (`Preprocessor.fit_transform`, horizons 1/6/12 h).


In [ ]:
from src.data.cdf_reader import read_goes_directory, read_wind_directory
from src.data.preprocessor import Preprocessor
from src.data.dataloader import make_dataloaders

with open(REPO_DIR / "configs/config.yaml") as f:
    base_config = yaml.safe_load(f)

print("forecast_horizons:", base_config["data"]["forecast_horizons"])
print("sequence_length:", base_config["data"]["sequence_length"])

goes = read_goes_directory(str(REPO_DIR / "datasets/goes"))
omni = read_wind_directory(str(REPO_DIR / "datasets/omni"))
raw = goes.join(omni, how="inner")
train_df, val_df, test_df = Preprocessor().fit_transform(raw)

seq_len = int(base_config["data"]["sequence_length"])
batch_size = int(base_config["training"].get("batch_size", 64))
storm_weight = float(base_config["training"].get("storm_weight", 12.0))

train_loader, val_loader, test_loader = make_dataloaders(
    train_df, val_df, test_df,
    seq_len=seq_len,
    batch_size=batch_size,
    storm_weight=storm_weight,
)
n_sw = int(next(iter(train_loader))["x_sw"].shape[-1])
print(f"splits train/val/test: {len(train_df)} / {len(val_df)} / {len(test_df)}")
print(f"n_sw_features={n_sw}  batches test={len(test_loader)}")


## 3. Optional demo train (1 model, 1 seed, few epochs)

Uses **`Trainer.fit(..., use_ensemble=False)`** — the real API.  
Does **not** replace the 15-seed paper campaign.


In [ ]:
from copy import deepcopy
from src.training.trainer import Trainer

DEMO_TRAIN = False   # set True to run a short local/Colab train
DEMO_EPOCHS = 3
DEMO_SEED = 42
DEMO_NAME = "storm_bz"  # or transformer / transformer_matched / lstm

if DEMO_TRAIN:
    cfg = deepcopy(base_config)
    cfg["training"]["seed"] = DEMO_SEED
    cfg["training"]["epochs"] = DEMO_EPOCHS
    cfg["training"]["checkpoint_dir"] = str(REPO_DIR / "checkpoints" / "_demo" / DEMO_NAME / f"seed_{DEMO_SEED}")
    cfg["training"]["log_dir"] = str(REPO_DIR / "logs" / "_demo" / DEMO_NAME / f"seed_{DEMO_SEED}")
    cfg.setdefault("model", {})

    if DEMO_NAME == "lstm":
        cfg["model_type"] = "lstm"
        cfg["ablation"] = "none"
    elif DEMO_NAME == "transformer":
        cfg["model_type"] = "transformer"
        cfg["match_storm_capacity"] = False
        cfg["ablation"] = "none"
    elif DEMO_NAME == "transformer_matched":
        cfg["model_type"] = "transformer"
        cfg["match_storm_capacity"] = True
        cfg["ablation"] = "none"
    elif DEMO_NAME == "storm_bz":
        cfg["model_type"] = "storm_physnet"
        cfg["model"]["gate_type"] = "bz"
        cfg["ablation"] = "none"
    else:
        raise ValueError(DEMO_NAME)

    torch.manual_seed(DEMO_SEED)
    np.random.seed(DEMO_SEED)
    trainer = Trainer(cfg)
    model = trainer.fit(train_loader, val_loader, n_sw_features=n_sw, use_ensemble=False)
    print("Demo train finished. Model params:", sum(p.numel() for p in model.parameters()))
else:
    print("DEMO_TRAIN=False — skipped. Set True for a short API check.")


## 4. Optional: evaluate one released checkpoint

Loads `checkpoints/<name>/seed_<id>/*_best.pt` with the matching `build_model` flags.


In [ ]:
from src.evaluation.metrics import prediction_efficiency, prediction_efficiency_pers

EVAL_ONE = True
EVAL_NAME = "storm_bz"
EVAL_SEED = 42

SPECS = {
    "lstm": dict(model_type="lstm", gate_type="bz", ablation="none", match=False, spectral=False),
    "transformer": dict(model_type="transformer", gate_type="bz", ablation="none", match=False, spectral=False),
    "transformer_matched": dict(model_type="transformer", gate_type="bz", ablation="none", match=True, spectral=False),
    "storm_bz": dict(model_type="storm_physnet", gate_type="bz", ablation="none", match=False, spectral=False),
    "storm_no_delay": dict(model_type="storm_physnet", gate_type="bz", ablation="no_delay", match=False, spectral=False),
    "storm_no_physics": dict(model_type="storm_physnet", gate_type="bz", ablation="no_physics", match=False, spectral=False),
    "storm_no_gate": dict(model_type="storm_physnet", gate_type="bz", ablation="no_bz_gate", match=False, spectral=False),
    "storm_cathode": dict(model_type="storm_physnet", gate_type="cathode_anode", ablation="none", match=False, spectral=False),
    "storm_cathode_spec": dict(model_type="storm_physnet", gate_type="cathode_anode", ablation="none", match=False, spectral=True),
    "storm_radiotrophic": dict(model_type="storm_physnet", gate_type="radiotrophic", ablation="none", match=False, spectral=False),
}

def make_cfg(spec):
    cfg = deepcopy(base_config)
    cfg["model_type"] = spec["model_type"]
    cfg["ablation"] = spec["ablation"]
    cfg["match_storm_capacity"] = bool(spec["match"])
    cfg.setdefault("model", {})
    cfg["model"]["gate_type"] = spec["gate_type"]
    cfg["model"]["use_spectral_head"] = bool(spec["spectral"])
    cfg["training"]["checkpoint_dir"] = str(REPO_DIR / "checkpoints" / "_eval_tmp")
    return cfg

def find_ckpt(name, seed):
    d = REPO_DIR / "checkpoints" / name / f"seed_{seed}"
    if not d.exists():
        return None
    pts = sorted(d.glob("*_best.pt"))
    return pts[0] if pts else None

@torch.no_grad()
def eval_pe(model, loader, device):
    model.eval()
    ys, ps, pers = [], [], []
    for batch in loader:
        x_sw = batch["x_sw"].to(device)
        x_flux = batch["x_flux"].to(device)
        y_persist = batch["y_persist"].to(device)
        try:
            out = model(x_sw, x_flux, y_persist)
        except TypeError:
            out = model(x_sw, x_flux)
        pred = out["flux_pred"] if isinstance(out, dict) else out
        ys.append(batch["y_flux"].numpy())
        ps.append(pred.cpu().numpy())
        pers.append(batch["y_persist"].numpy())
    y, p, yp = map(np.concatenate, (ys, ps, pers))
    return {
        "PE_1h": float(prediction_efficiency(y[:, 0], p[:, 0])),
        "PE_6h": float(prediction_efficiency(y[:, 1], p[:, 1])),
        "PE_12h": float(prediction_efficiency(y[:, 2], p[:, 2])),
        "PE_pers_1h": float(prediction_efficiency_pers(y[:, 0], p[:, 0], yp[:, 0])),
    }

if EVAL_ONE:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    pt = find_ckpt(EVAL_NAME, EVAL_SEED)
    if pt is None:
        print(f"Checkpoint not found for {EVAL_NAME} seed {EVAL_SEED} under checkpoints/")
        print("Skip or download released checkpoints from the repo.")
    else:
        cfg = make_cfg(SPECS[EVAL_NAME])
        trainer = Trainer(cfg)
        model = trainer.build_model(n_sw)
        state = torch.load(pt, map_location=device, weights_only=True)
        model.load_state_dict(state, strict=True)
        model.to(device)
        metrics = eval_pe(model, test_loader, device)
        print(f"{EVAL_NAME} seed={EVAL_SEED} from {pt.name}")
        print({k: round(v, 4) for k, v in metrics.items()})
else:
    print("EVAL_ONE=False — skipped")


## 5. GRASP summary (from official CSV)

Full zero-shot + fine-tune protocol is in the archived Kaggle GRASP script; results are frozen in `results/grasp_summary.csv`.


In [ ]:
gs = pd.read_csv(RES / "grasp_summary.csv")
cols = [c for c in gs.columns if "PE_6h" in c or "PE_12h" in c or c == "name" or c == "n"]
print(gs[cols].round(3).to_string(index=False))
print("\nPaper table (STORM-Bz):")
print(pd.read_csv(RES / "table_grasp_storm_bz.csv").to_string(index=False))


## 6. Plot official means (optional figure check)


In [ ]:
plot_names = [n for n in ["lstm", "transformer", "storm_bz", "transformer_matched", "storm_cathode", "storm_radiotrophic"] if n in set(main["name"])]
labels = {
    "lstm": "LSTM", "transformer": "Transformer", "storm_bz": "STORM-Bz",
    "transformer_matched": "TF matched", "storm_cathode": "RDG", "storm_radiotrophic": "SDG",
}
sub = main.set_index("name").loc[plot_names]
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(plot_names))
w = 0.25
for i, h in enumerate(["PE_1h", "PE_6h", "PE_12h"]):
    ax.bar(x + (i - 1) * w, sub[h].values, w, label=h.replace("PE_", ""))
ax.set_xticks(x)
ax.set_xticklabels([labels.get(n, n) for n in plot_names], rotation=15, ha="right")
ax.set_ylabel(r"PE$_{clim}$ (seed mean)")
ax.set_title("Official table_main_means (15 seeds)")
ax.legend()
ax.set_ylim(0.82, 1.0)
fig.tight_layout()
out = REPO_DIR / "figures" / "fig_horizon_pe_from_table.png"
out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out, dpi=150)
plt.show()
print("saved", out)


## 7. Reviewer map

| Claim in paper | Artifact |
|----------------|----------|
| GOES seed-mean PE | `results/table_main_means.csv` |
| Bootstrap / std | `results/table_means_bootstrap_ci.csv`, `table_main_stats.csv` |
| True bagging | `results/table_bagged.csv` |
| Ensemble α* | `results/ensemble_summary.json` |
| GRASP 0.740→0.841 | `results/table_grasp_storm_bz.csv` |
| Checkpoints | `checkpoints/<model>/seed_{42..56}/` |
| Full multi-seed train | Kaggle multi-account campaign (not this notebook) |

**Do not use** `trainer.train` — it does not exist. Use `Trainer.fit` or load checkpoints.
